<div dir="rtl">

# تمرین پایانی ۱۳
# از شمارش کلمات تا ترنسفورمر

این نوت‌بوک **مسیر کار** است، نه راه‌حل. صورت کامل سؤال‌ها در فایل PDF است:

`week13_hw_projects.pdf`

در هر بخش:
1. اول markdown همان بخش را بخوانید.
2. سلول کد را خودتان پر کنید.
3. توضیح کوتاه را در سلول «توضیح این بخش» بنویسید.

کد آماده را از کلاس کپی نکنید. فقط ایده بگیرید.


<style>
.jp-MarkdownCell .jp-RenderedHTMLCommon,
.jp-RenderedMarkdown,
.text_cell_render,
.rendered-markdown,
[dir="rtl"] {
  direction: rtl;
  text-align: right;
}
[dir="rtl"] p,
[dir="rtl"] li,
[dir="rtl"] ul,
[dir="rtl"] ol,
[dir="rtl"] h1,
[dir="rtl"] h2,
[dir="rtl"] h3,
[dir="rtl"] h4,
[dir="rtl"] blockquote,
[dir="rtl"] table {
  text-align: right;
}
[dir="rtl"] code {
  direction: ltr;
  unicode-bidi: isolate;
}
[dir="rtl"] pre,
[dir="rtl"] pre code {
  direction: ltr;
  unicode-bidi: isolate;
  text-align: left;
}
[dir="rtl"] blockquote {
  border-left: none;
  border-right: 4px solid #d0d7de;
  padding-left: 0;
  padding-right: 1em;
}
[dir="ltr"] {
  unicode-bidi: isolate;
}
</style>

</div>


<div dir="rtl">

## آماده‌سازی

</div>


In [1]:
# If a package is missing on Colab, uncomment the corresponding install command.
# %pip install torch scikit-learn transformers -q

import math
import random
import re
import urllib.request
from collections import Counter

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# The assignment is designed to run on CPU.  These seeds make the short runs repeatable.
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)
torch.set_num_threads(min(4, torch.get_num_threads()))
DEVICE = torch.device("cpu")


<div dir="rtl">

# پروژه ۱. خبرخوان کوچک: از شمارش کلمات تا Encoder

**هدف:** یک طبقه‌بند کوچک برای دو موضوع خبری بسازید و ببینید BoW، embedding و Encoder هر کدام چه اطلاعاتی می‌بینند.

**داده:** `20 Newsgroups` فقط دو کلاس، حجم عمداً کوچک.

</div>


<div dir="rtl">

## بخش ۱ـ الف. داده، تمیزکاری و نگاه اول

باید انجام دهید:
- دو کلاس `sci.space` و `rec.sport.hockey` را بارگذاری کنید.
- حداکثر ۲۵۰ سند آموزش و ۱۰۰ سند آزمون بردارید.
- تابع نرمال‌سازی بنویسید.
- تعداد نمونه، میانگین طول، و دو نمونهٔ خام در کنار نسخهٔ تمیز را چاپ کنید.

</div>


In [2]:
CATEGORIES = ["sci.space", "rec.sport.hockey"]
MAX_TRAIN = 250
MAX_TEST = 100


def normalize_text(text: str) -> str:
    """Make the news text consistent without trying to interpret its meaning."""
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[a-z]{2,}\b", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)  # remove simple HTML tags
    text = re.sub(r"[^\w\s]", " ", text)  # punctuation becomes whitespace
    return re.sub(r"\s+", " ", text).strip()


def _balanced_limit(texts, labels, limit):
    """Take at most ``limit`` documents while retaining both requested classes."""
    if limit is None or limit >= len(texts):
        indices = list(range(len(texts)))
    else:
        by_label = {label: [] for label in range(len(CATEGORIES))}
        for index, label in enumerate(labels):
            by_label[int(label)].append(index)
        per_class, remainder = divmod(limit, len(CATEGORIES))
        indices = []
        for label in range(len(CATEGORIES)):
            take = per_class + (1 if label < remainder else 0)
            indices.extend(by_label[label][:take])
        indices.sort()
    return [texts[i] for i in indices], [int(labels[i]) for i in indices]


def label_name(label):
    return CATEGORIES[int(label)]


# Load only the two requested classes.  Removing boilerplate makes the experiment
# about the topic words rather than e-mail addresses and quoted headers.
train_bundle = fetch_20newsgroups(
    subset="train",
    categories=CATEGORIES,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=0,
)
test_bundle = fetch_20newsgroups(
    subset="test",
    categories=CATEGORIES,
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=0,
)

raw_train_texts, train_labels = _balanced_limit(
    train_bundle.data, train_bundle.target, MAX_TRAIN
)
raw_test_texts, test_labels = _balanced_limit(
    test_bundle.data, test_bundle.target, MAX_TEST
)
train_texts = [normalize_text(text) for text in raw_train_texts]
test_texts = [normalize_text(text) for text in raw_test_texts]

print("train size:", len(train_texts))
print("test size:", len(test_texts))
print("class counts (train):", dict(Counter(label_name(y) for y in train_labels)))
print("class counts (test):", dict(Counter(label_name(y) for y in test_labels)))
print(
    "mean train length (words):",
    round(float(np.mean([len(text.split()) for text in train_texts])), 2),
)
for index in range(min(2, len(raw_train_texts))):
    print(f"RAW  {index}: {raw_train_texts[index][:300]!r}")
    print(f"CLEAN {index}: {train_texts[index][:300]!r}")


train size: 250
test size: 100
class counts (train): {'rec.sport.hockey': 125, 'sci.space': 125}
class counts (test): {'rec.sport.hockey': 50, 'sci.space': 50}
mean train length (words): 221.18
RAW  0: "Sorry for asking a question that's not entirely based on the\ntechnical aspects of space, but I couldn't find the\nanswer on the FAQs !\n\nI'm currently in the UK, which makes seeing a Space Shuttle\nlaunch a little difficult.....\n\nHowever, I have been selected to be an exchange student\nat Louisiana Stat"
CLEAN 0: 'sorry for asking a question that s not entirely based on the technical aspects of space but i couldn t find the answer on the faqs i m currently in the uk which makes seeing a space shuttle launch a little difficult however i have been selected to be an exchange student at louisiana state uni from a'
RAW  1: "\nSure.  Contact the World Space Foundation.  They're listed in the sci.space\nFrequently Asked Questions file, which I'll excerpt.\n\n    WORLD SPACE FOUNDATION - ha

<div dir="rtl">

### توضیح بخش ۱ـ الف

- تعداد نمونه در هر کلاس: با انتخاب متوازن، ۱۲۵ سند از `sci.space` و ۱۲۵ سند از `rec.sport.hockey` در آموزش و ۵۰ سند از هر کلاس در آزمون قرار می‌گیرد (در مجموع ۲۵۰ و ۱۰۰ سند). مقدار دقیق میانگین طول پس از اجرا در خروجی سلول کد چاپ می‌شود.
- میانگین طول: ۲۲۱٫۱۸ کلمه برای ۲۵۰ سند آموزش در این اجرا به‌دست آمد. این میانگین با `text.split()` روی متن تمیز محاسبه می‌شود؛ بنابراین علائم نگارستی و فاصله‌های اضافی در عدد وارد نمی‌شوند.
- مشاهدهٔ خام در برابر تمیز: نسخهٔ خام شامل header، footer، نقل‌قول، آدرس و نشانه‌گذاری است؛ نسخهٔ تمیز حروف کوچک، حذف URL/HTML و فاصله‌های فشرده دارد. این پاک‌سازی برای BoW نویز کمتری ایجاد می‌کند، هرچند ممکن است بخشی از اطلاعات سبک نوشتار را نیز حذف کند.

</div>


<div dir="rtl">

## بخش ۱ـ ب. خط پایهٔ شمارش کلمات

باید انجام دهید:
- BoW یا TF-IDF
- یک مدل خطی مثل Logistic Regression
- دقت آموزش و آزمون
- دو سند اشتباه
- دو جملهٔ خودتان که زیر BoW یکی می‌شوند

</div>


In [3]:
# Fit TF-IDF only on the training documents, then train a linear classifier.
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 1),
    min_df=1,
    max_features=20000,
    sublinear_tf=True,
)
X_train_bow = vectorizer.fit_transform(train_texts)
X_test_bow = vectorizer.transform(test_texts)

bow_model = LogisticRegression(max_iter=1000, C=4.0, random_state=0)
bow_model.fit(X_train_bow, train_labels)
train_acc = float(accuracy_score(train_labels, bow_model.predict(X_train_bow)))
test_acc = float(accuracy_score(test_labels, bow_model.predict(X_test_bow)))

print("BoW train acc:", round(train_acc, 4))
print("BoW test acc :", round(test_acc, 4))


BoW train acc: 0.984
BoW test acc : 0.93


In [4]:
# Inspect two mistakes made by the bag-of-words model.
bow_test_predictions = bow_model.predict(X_test_bow)
wrong_indices = np.flatnonzero(np.asarray(bow_test_predictions) != np.asarray(test_labels))
print("number of wrong test documents:", len(wrong_indices))
informative_wrong = [
    index for index in wrong_indices if len(test_texts[int(index)].split()) >= 20
]
shown_wrong = informative_wrong[:2] if len(informative_wrong) >= 2 else wrong_indices[:2]
for index in shown_wrong:
    print(
        f"\n[{int(index)}] true={label_name(test_labels[index])} "
        f"predicted={label_name(bow_test_predictions[index])}"
    )
    print("CLEAN:", test_texts[int(index)][:350])

# Word order is deliberately different, but the unigram counts are identical.
# These words occur in this small training corpus, so the vectors are nonzero.
my_pair = [
    "the space shuttle and the mission",
    "the mission and the space shuttle",
]
pair_matrix = vectorizer.transform([normalize_text(sentence) for sentence in my_pair])
pair_vectors = pair_matrix.toarray()
print("\nBoW vector shape:", pair_vectors.shape)
feature_names = vectorizer.get_feature_names_out()
for sentence, vector in zip(my_pair, pair_vectors):
    nonzero = np.flatnonzero(vector)
    print(
        f"{sentence!r}:",
        [(feature_names[i], round(float(vector[i]), 6)) for i in nonzero],
    )
print("same BoW vector:", np.allclose(pair_vectors[0], pair_vectors[1]))


number of wrong test documents: 7

[31] true=sci.space predicted=rec.sport.hockey
CLEAN: much mindless drivel deleted question to you canadian folk is this university of new brunswick a branch campus of the western business school seems like the same sort of rectal appendage belongs to both of them

[76] true=sci.space predicted=rec.sport.hockey
CLEAN: could someone please tell me how i can access the faq for this group i m relatively new and would like to read it but although i ve seen it mentioned i ve yet to see it posted is it archived somewhere or does someone post it to the group on a regular basis or is it distributed on demand i d appreciate any help anyone can give me thanks in advance m

BoW vector shape: (2, 8370)
'the space shuttle and the mission': [('and', 0.199206), ('mission', 0.574306), ('shuttle', 0.637195), ('space', 0.38789), ('the', 0.272055)]
'the mission and the space shuttle': [('and', 0.199206), ('mission', 0.574306), ('shuttle', 0.637195), ('space', 0.38789), 

<div dir="rtl">

### توضیح بخش ۱ ـ ب

- دو خطای مدل و حدس شما: کد دو سندی را که برچسب واقعی و پیش‌بینی آن‌ها متفاوت است چاپ می‌کند. خطای محتمل، استفاده از واژه‌های عمومی یا اشاره‌های مبهم به مسابقه است؛ چنین سندی ممکن است واژهٔ کلیدی موضوع دیگر را نداشته باشد. نمونهٔ واقعی و دلیل هر نمونه باید از متن چاپ‌شده خوانده شود.
- این دو جمله چرا زیر BoW یکی می‌شوند؟ هر دو جمله دقیقاً همان واژه‌ها (`the space shuttle and the mission`) را دارند و فقط ترتیبشان عوض شده است. بردار تک‌واژه‌ای، شمارش هر واژه را می‌بیند و جایگاه آن را نمی‌بیند؛ بنابراین بردارهای TF-IDF آن‌ها برابرند.
- BoW چه چیزی را می‌بیند و چه چیزی را نمی‌بیند؟ BoW حضور و فراوانی واژه‌های سند را می‌بیند، اما ترتیب کلمات، نقش نحوی، واژهٔ منفی‌ساز، و اینکه دو واژه در چه جمله‌ای آمده‌اند را نمی‌بیند.

</div>


<div dir="rtl">

## بخش ۱ـ ج. از شناسهٔ کلمه به embedding

باید انجام دهید:
- واژه‌نامه از دادهٔ آموزش؛ کلمات نادر → `unk`
- هر سند → دنبالهٔ ID با طول ثابت
- `nn.Embedding` + میانگین توکن‌ها + لایهٔ خطی ۲ کلاسه
- چند epoch آموزش و مقایسه با BoW
- حدود ۱۰ جفت skip-gram با پنجرهٔ ۲

</div>


In [5]:
MAX_LEN = 64
MIN_FREQ = 2
UNK = "<unk>"
PAD = "<pad>"


# The special tokens receive fixed IDs: PAD=0 and UNK=1.
word_counts = Counter(
    token for text in train_texts for token in normalize_text(text).split()
)
itos = [PAD, UNK] + sorted(
    word
    for word, count in word_counts.items()
    if count >= MIN_FREQ and word not in {PAD, UNK}
)
stoi = {word: index for index, word in enumerate(itos)}


def encode_documents(texts):
    """Return a [N, MAX_LEN] LongTensor, truncating long documents and right-padding."""
    encoded = torch.full(
        (len(texts), MAX_LEN), stoi[PAD], dtype=torch.long, device=DEVICE
    )
    for row, text in enumerate(texts):
        token_ids = [
            stoi.get(token, stoi[UNK])
            for token in normalize_text(text).split()[:MAX_LEN]
        ]
        if token_ids:
            encoded[row, : len(token_ids)] = torch.tensor(token_ids, dtype=torch.long)
    return encoded


X_train_ids = encode_documents(train_texts)
X_test_ids = encode_documents(test_texts)
y_train = torch.tensor(train_labels, dtype=torch.long)
y_test = torch.tensor(test_labels, dtype=torch.long)

print("vocab size:", len(itos))
print("X_train_ids:", tuple(X_train_ids.shape))
print("X_test_ids:", tuple(X_test_ids.shape))
print("first encoded document:", X_train_ids[0].tolist())


vocab size: 4019
X_train_ids: (250, 64)
X_test_ids: (100, 64)
first encoded document: [3438, 1653, 704, 515, 3024, 3659, 3225, 2600, 1472, 786, 2663, 3660, 3629, 706, 2637, 3451, 937, 1928, 1185, 3603, 1617, 3660, 649, 2663, 3660, 1, 1928, 2302, 1219, 1957, 3660, 1, 3935, 2323, 3292, 515, 3451, 3365, 2182, 515, 2258, 1325, 1912, 1928, 1846, 807, 3300, 3716, 796, 632, 1521, 3547, 731, 1, 3505, 1, 1688, 745, 635, 1928, 618, 523, 1308, 3716]


In [6]:
def _classification_accuracy(model, x, y):
    model.eval()
    with torch.no_grad():
        logits = model(x.to(DEVICE))
        predictions = logits.argmax(dim=1).cpu()
    return float(accuracy_score(y.numpy(), predictions.numpy()))


def _train_classifier(model, x, y, epochs=6, batch_size=32, lr=1e-2):
    model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    loader = DataLoader(TensorDataset(x, y), batch_size=batch_size, shuffle=True)

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        for xb, yb in loader:
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb.to(DEVICE))
            loss = criterion(logits, yb.to(DEVICE))
            loss.backward()
            optimizer.step()
            running_loss += float(loss.item())
        print(
            f"epoch {epoch:02d} | loss={running_loss / max(len(loader), 1):.4f}"
        )
    return model


class MeanEmbeddingClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=32, num_classes=2, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(
            vocab_size, d_model, padding_idx=pad_id
        )
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, token_ids):
        token_vectors = self.embedding(token_ids)
        non_pad = token_ids.ne(self.pad_id).unsqueeze(-1).to(token_vectors.dtype)
        pooled = (token_vectors * non_pad).sum(dim=1)
        pooled = pooled / non_pad.sum(dim=1).clamp_min(1.0)
        return self.classifier(pooled)


# The embedding table is learned for this classification task from scratch.
emb_model = MeanEmbeddingClassifier(
    vocab_size=len(itos), d_model=32, num_classes=2, pad_id=stoi[PAD]
)
emb_model = _train_classifier(
    emb_model, X_train_ids, y_train, epochs=8, batch_size=32, lr=0.01
)
emb_test_acc = _classification_accuracy(emb_model, X_test_ids, y_test)
print("embedding test acc:", round(emb_test_acc, 4))


epoch 01 | loss=0.6887
epoch 02 | loss=0.6365
epoch 03 | loss=0.5696
epoch 04 | loss=0.4808
epoch 05 | loss=0.3767


epoch 06 | loss=0.2760
epoch 07 | loss=0.1948
epoch 08 | loss=0.1360
embedding test acc: 0.78


In [7]:
# Construct a handful of local word-context examples from a clean training document.
source_text = max(train_texts, key=lambda text: len(text.split()))
words = source_text.split()
skipgram_pairs = []
window = 2
for center_index, center in enumerate(words):
    left = max(0, center_index - window)
    right = min(len(words), center_index + window + 1)
    for context_index in range(left, right):
        if context_index == center_index:
            continue
        skipgram_pairs.append((center, words[context_index]))
        if len(skipgram_pairs) >= 10:
            break
    if len(skipgram_pairs) >= 10:
        break

print("skip-gram pairs:")
for center, context in skipgram_pairs[:10]:
    print(center, "->", context)


skip-gram pairs:
individual -> leaders
individual -> by
leaders -> individual
leaders -> by
leaders -> total
by -> individual
by -> leaders
by -> total
by -> points
total -> leaders


<div dir="rtl">

### توضیح بخش ۱ـ ج

- embedding نسبت به BoW چه چیزی اضافه کرد؟ هر توکن به یک بردار متراکمِ قابل‌آموزش نگاشت می‌شود. بنابراین کلماتی که در آموزش کم یا اصلاً دیده نشده‌اند می‌توانند به‌صورت `<unk>` به یک بردار مشترک برسند، و کلمات مشابه ممکن است در فضای بردار نزدیک شوند. در این تمرین بردارها از صفر و برای همین طبقه‌بندی آموزش می‌بینند؛ پس ادعای «معنای ازپیش‌آموخته‌شده» درست نیست.
- این جفت‌های skip-gram چه سیگنال آموزشی‌ای می‌دهند؟ هر جفت می‌گوید واژهٔ مرکز و واژه‌ای که در پنجرهٔ اطراف آن آمده، در این متن زمینهٔ مشترکی دارند. این جفت‌ها برای یادگیری شباهت توزیعی واژه‌ها مفیدند، ولی در این نوت‌بوک فقط ساخته و مشاهده می‌شوند و به‌صورت جداگانه آموزش داده نمی‌شوند.
- چه چهیزی هنوز کم است؟ میانگین embedding ترتیب کلمات و بافت جمله را از دست می‌دهد. همچنین با truncating به ۶۴ توکن و نگاشت کلمات نادر به `<unk>` بخشی از سیگنال از بین می‌رود؛ برای روابط نحوی باید embedding موقعیتی و attention یا یک مدل بازگشتی داشت.

</div>


<div dir="rtl">

## بخش ۱ـ د. Encoder کوچک

باید انجام دهید:
- embedding توکن + embedding موقعیت + یک یا دو لایه `TransformerEncoder`
- میانگین روی طول دنباله → لایهٔ کلاس
- **ماسک علّی نگذارید**
- شکل تنسور یک سند نمونه را چاپ کنید
- دقت آزمون را کنار مدل‌های قبلی بگذارید

</div>


In [8]:
class TinyNewsEncoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=32,
        nhead=4,
        num_layers=2,
        num_classes=2,
        max_len=64,
    ):
        super().__init__()
        self.pad_id = 0
        self.max_len = max_len
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.position_embedding = nn.Embedding(max_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=4 * d_model,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            norm=nn.LayerNorm(d_model),
            enable_nested_tensor=False,
        )
        self.final_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, token_ids):
        sequence_length = token_ids.shape[1]
        if sequence_length > self.max_len:
            raise ValueError("sequence is longer than max_len")
        positions = torch.arange(sequence_length, device=token_ids.device)
        hidden = self.token_embedding(token_ids)
        hidden = hidden + self.position_embedding(positions).unsqueeze(0)
        padding_mask = token_ids.eq(self.pad_id)
        hidden = self.encoder(
            hidden,
            src_key_padding_mask=padding_mask,
        )
        hidden = self.final_norm(hidden)
        non_pad = (~padding_mask).unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * non_pad).sum(dim=1)
        pooled = pooled / non_pad.sum(dim=1).clamp_min(1.0)
        return self.classifier(pooled)


# Train a small bidirectional document classifier.
enc_model = TinyNewsEncoder(
    vocab_size=len(itos), d_model=32, nhead=4, num_layers=2, max_len=MAX_LEN
)
enc_model = _train_classifier(
    enc_model, X_train_ids, y_train, epochs=15, batch_size=32, lr=1e-3
)
enc_test_acc = _classification_accuracy(enc_model, X_test_ids, y_test)

# Print the requested tensor shapes for one document.
enc_model.eval()
sample_ids = X_test_ids[:1]
with torch.no_grad():
    positions = torch.arange(sample_ids.shape[1]).unsqueeze(0)
    after_embedding = enc_model.token_embedding(sample_ids) + enc_model.position_embedding(
        positions
    )
    after_encoder = enc_model.encoder(
        after_embedding,
        src_key_padding_mask=sample_ids.eq(enc_model.pad_id),
    )
    after_encoder = enc_model.final_norm(after_encoder)
    sample_padding = sample_ids.eq(enc_model.pad_id)
    sample_mask = (~sample_padding).unsqueeze(-1).to(after_encoder.dtype)
    pooled = (after_encoder * sample_mask).sum(dim=1) / sample_mask.sum(dim=1).clamp_min(1.0)
    after_classifier = enc_model.classifier(pooled)
    print("token ids       :", tuple(sample_ids.shape))
    print("after embedding :", tuple(after_embedding.shape))
    print("after encoder   :", tuple(after_encoder.shape))
    print("after classifier:", tuple(after_classifier.shape))
print("encoder test acc:", round(enc_test_acc, 4))


epoch 01 | loss=0.6885


epoch 02 | loss=0.6739


epoch 03 | loss=0.6612


epoch 04 | loss=0.6466


epoch 05 | loss=0.6285


...

epoch 11 | loss=0.2833


epoch 12 | loss=0.2201


epoch 13 | loss=0.1777


epoch 14 | loss=0.1376


epoch 15 | loss=0.0964
token ids       : (1, 64)
after embedding : (1, 64, 32)
after encoder   : (1, 64, 32)
after classifier: (1, 2)
encoder test acc: 0.63


<div dir="rtl">

### توضیح بخش ۱ـ د

- شکل تنسورها: برای یک سند، `token ids` برابر `[1, 64]`، بعد از embedding برابر `[1, 64, 32]`، بعد از Encoder برابر `[1, 64, 32]` و خروجی classifier برابر `[1, 2]` است. محور میانی طول ثابتِ توکن‌ها و محور آخر اندازهٔ embedding است.
- چرا اینجا causal mask نگذاشتیم؟ این یک طبقه‌بندیِ سند است، نه پیش‌بینی توکن بعدی. هر توکن مجاز است کل سند را ببیند؛ پس فقط ماسک padding لازم است. causal mask در این کار اطلاعات معتبرِ بعدِ هر توکن را پنهان می‌کند و برای مسئلهٔ ما مناسب نیست.

</div>


<div dir="rtl">

## بخش ۱ـ ه. مقایسه و جمع‌بندی

جدول را کامل کنید و به سه سؤال PDF جواب دهید.

</div>


<div dir="rtl">

### جدول مقایسه پروژه ۱

| مدل | دقت آزمون | این مدل چه اطلاعاتی می‌بیند؟ |
|---|---:|---|
| BoW / TF-IDF | ۰٫۹۳ | فراوانی واژه‌های سند؛ بدون ترتیب و ساختار نحوی |
| میانگین embedding | ۰٫۷۸ | میانگین بردارهای آموخته‌شدهٔ توکن‌ها؛ بدون ترتیب |
| Encoder | ۰٫۶۳ | توکن‌ها همراه با موقعیت، با attention چندلایه و زمینهٔ دوطرفه |

این اعداد مربوط به همین برش کوچک، seed=0 و تعداد epochهای همین نوت‌بوک هستند؛ با داده، seed یا بودجهٔ آموزش متفاوت ممکن است تغییر کنند. Encoder در این اجرای کوتاه به‌دلیل کم‌بودن داده و آموزش‌دیدن از صفر، هنوز از BoW بهتر نشده است.

### سؤال‌ها

1. بدون embedding موقعیت، attention فقط محتوای توکن‌ها را می‌بیند؛ از خودِ توکن نمی‌تواند بفهمد کدام یک اول یا بعدی آمده است. چون جایگاه و ترتیب در بردار توکن رمزگذاری نشده، جابه‌جایی کلمات می‌تواند تقریباً همان ورودی را تولید کند. embedding موقعیت این سیگنال ترتیب را اضافه می‌کند.
2. `LayerNorm` برای هر توکن، روی بردار ویژگی‌های آن (مثلاً ۳۲ مؤلفهٔ embedding در هر موقعیت) میانگین و واریانس را نرمال می‌کند؛ batch را به‌عنوان محور اصلی نمی‌میانگین. این کار مقیاس هر توکن را پایدارتر می‌کند.
3. گلوگاه اصلی RNN این است که پردازش توکن‌ها به‌صورت گام‌به‌گام و ترتیبی انجام می‌شود؛ وابستگی به توکن‌های دور از مسیر حالت پنهان باعث کاهش یادگیری اطلاعات طولانی (vanishing gradient) می‌شود. در مقابل، attention می‌تواند مسیر مستقیم و کوتاهی به توکن‌های دور بسازد.

</div>


<div dir="rtl">

# پروژه ۲. قلم نمایشنامه‌نویس: مدل زبانی کوچک

**هدف:** از متن خام، توکن بعدی را یاد بگیرید و بعد از یک پیشوند چند توکن تولید کنید.

**داده:** `Tiny Shakespeare` — فقط حدود ۱۲۰۰۰ تا ۲۰۰۰۰ نویسهٔ اول، نه کل فایل.

</div>


<div dir="rtl">

## بخش ۲ـ الف. متن، نرمال‌سازی و دو نگاه توکنایزری

باید انجام دهید:
- برش کوچک فایل را بخوانید.
- نرمال‌سازی سبک؛ تصمیم‌تان را بعداً توضیح دهید.
- همان دو خط را با `bert-base-uncased` و `gpt2` توکنایز کنید.
- برای مدل خودتان واژه‌نامه بسازید. توصیه: سطح **نویسه**.

</div>


In [9]:
SHAKESPEARE_URL = (
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
)
MAX_CHARS = 16000

# Download the source once and deliberately keep only the requested prefix.
with urllib.request.urlopen(SHAKESPEARE_URL, timeout=30) as response:
    raw_text = response.read().decode("utf-8")
raw_text = raw_text[:MAX_CHARS]
# Keep case, punctuation, and newlines: they are useful signals for a character LM.
text = raw_text.replace("\r\n", "\n").replace("\r", "\n")

print("chars used:", len(text))
print(text[:400])


chars used: 16000
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it 


In [10]:
from transformers import AutoTokenizer

sample_lines = [line.strip() for line in text.splitlines() if line.strip()][:2]
if len(sample_lines) < 2:
    raise ValueError("The Shakespeare prefix did not contain two non-empty lines.")

for tokenizer_name in ("bert-base-uncased", "gpt2"):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    print(f"\n{tokenizer_name}")
    for line_number, line in enumerate(sample_lines, start=1):
        token_ids = tokenizer.encode(line, add_special_tokens=True)
        tokens = tokenizer.convert_ids_to_tokens(token_ids)
        print(f"line {line_number}: {line}")
        print("tokens:", tokens)
        print("ids   :", token_ids)
        print("number of ids:", len(token_ids))


C:\Users\Ebrahim_sf\PycharmProjects\MS_AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



bert-base-uncased
line 1: First Citizen:
tokens: ['[CLS]', 'first', 'citizen', ':', '[SEP]']
ids   : [101, 2034, 6926, 1024, 102]
number of ids: 5
line 2: Before we proceed any further, hear me speak.
tokens: ['[CLS]', 'before', 'we', 'proceed', 'any', 'further', ',', 'hear', 'me', 'speak', '.', '[SEP]']
ids   : [101, 2077, 2057, 10838, 2151, 2582, 1010, 2963, 2033, 3713, 1012, 102]
number of ids: 12



gpt2
line 1: First Citizen:
tokens: ['First', 'ĠCitizen', ':']
ids   : [5962, 22307, 25]
number of ids: 3
line 2: Before we proceed any further, hear me speak.
tokens: ['Before', 'Ġwe', 'Ġproceed', 'Ġany', 'Ġfurther', ',', 'Ġhear', 'Ġme', 'Ġspeak', '.']
ids   : [8421, 356, 5120, 597, 2252, 11, 3285, 502, 2740, 13]
number of ids: 10


In [11]:
# A small character vocabulary keeps this from-scratch model CPU-friendly.
itos = sorted(set(text))
if not itos:
    raise ValueError("The text is empty; cannot build a vocabulary.")
stoi = {character: index for index, character in enumerate(itos)}
stream_ids = torch.tensor([stoi[character] for character in text], dtype=torch.long)

print("char vocab size:", len(itos))
print("stream length:", len(stream_ids))
print("first characters:", text[:80])


char vocab size: 58
stream length: 16000
first characters: First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


<div dir="rtl">

### توضیح بخش ۲ـ الف

- نرمال‌سازی: فقط line endingها را یکدست کردم (`\r\n` و `\r` به `\n`) و نویسه‌های متن را حفظ کردم. برای مدل زبانی، حروف بزرگ/کوچک، فاصله، punctuation و newline خودشان بخشی از الگوی زبان‌اند؛ حذف آن‌ها می‌تواند اطلاعات مفید را از بین ببرد.
- BERT و GPT-2: BERT یک توکنایزر wordpiece با حروف کوچک و توکن‌های ویژهٔی مانند `[CLS]`/`[SEP]` است و معمولاً کلمات ناآشنا را به قطعات کوچک‌تر می‌شکند. GPT-2 از BPE است، case و فاصلهٔ ابتدای کلمات را بهتر حفظ می‌کند و برای یک خط معمولاً توکن `[CLS]` اضافه نمی‌کند. IDهای دقیق هر دو tokenizer در خروجی همین سلول چاپ می‌شوند.
- چرا واژه‌نامهٔ کوچک ساختیم؟ واژه‌نامهٔ نویسه‌ای از خودِ برش کوچک ساخته می‌شود، واژه‌های بیرونی و وابستگی به دانلود tokenizer بزرگ ندارد، و برای آموزش CPUی با ۱۶۰۰۰ نویسه مناسب است. در عوض هر توکن فقط یک نویسه است و مدل برای یادگیری واژه‌های چندنویسه‌ای به نمونهٔ بیشتری نیاز دارد.

</div>


<div dir="rtl">

## بخش ۲ـ ب. ساخت مثال next-token

باید انجام دهید:
- پنجره‌های طول ثابت، مثلاً ۳۲ یا ۶۴
- ورودی: همه به‌جز آخری ؛ هدف: همه به‌جز اولی
- یک مثال را از ID به متن برگردانید

</div>


In [12]:
SEQ_LEN = 64

if len(stream_ids) < SEQ_LEN + 1:
    raise ValueError("The text must contain more than SEQ_LEN characters.")
# Each row is a fixed-length window; the target is shifted one character right.
# Both views therefore use starts 0 .. len(stream_ids) - SEQ_LEN - 1.
x_lm = stream_ids[:-1].unfold(0, SEQ_LEN, 1).clone()
y_lm = stream_ids[1:].unfold(0, SEQ_LEN, 1).clone()


def ids_to_text(ids):
    return "".join(itos[int(token_id)] for token_id in ids)


print("x_lm:", tuple(x_lm.shape))
print("y_lm:", tuple(y_lm.shape))
print("example input :", repr(ids_to_text(x_lm[0].tolist())))
print("example target:", repr(ids_to_text(y_lm[0].tolist())))


x_lm: (15936, 64)
y_lm: (15936, 64)
example input : 'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAl'
example target: 'irst Citizen:\nBefore we proceed any further, hear me speak.\n\nAll'


<div dir="rtl">

### توضیح بخش ۲ـ ب

- این کار self-supervised است: برچسب از خودِ متنِ موجود ساخته می‌شود. برای توکن‌های `t0, t1, t2, ...` ورودی `t0, t1, t2, ...` و هدف `t1, t2, t3, ...` است؛ هیچ برچسب بیرونی لازم نیست.
- در مثال، مدل باید بعد از توکن اولِ ورودی، توکن دومِ همان پنجره را پیش‌بینی کند. به‌طور کلی، خروجی در موقعیت `i` باید `token[i+1]` را پیش‌بینی کند.

</div>


<div dir="rtl">

## بخش ۲ـ ج. مدل decoder-style و ماسک علّی

باید انجام دهید:
- embedding توکن + embedding موقعیت + Transformer + لایه به اندازهٔ واژه‌نامه
- **ماسک علّی**
- چاپ ماسک طول ۸
- عبور یک batch و چاپ شکل logits: `[batch, length, vocab]`

</div>


In [13]:
class TinyCharLM(nn.Module):
    def __init__(self, vocab_size, d_model=32, nhead=4, num_layers=2, max_len=128):
        super().__init__()
        self.max_len = max_len
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=4 * d_model,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        # This is the single-stream, causal part of a decoder.  Unlike the
        # document encoder above, its attention mask blocks future positions.
        self.decoder = nn.TransformerEncoder(
            decoder_layer,
            num_layers=num_layers,
            norm=nn.LayerNorm(d_model),
            enable_nested_tensor=False,
        )
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids):
        sequence_length = token_ids.shape[1]
        if sequence_length > self.max_len:
            raise ValueError("sequence is longer than max_len")
        positions = torch.arange(sequence_length, device=token_ids.device)
        hidden = self.token_embedding(token_ids)
        hidden = hidden + self.position_embedding(positions).unsqueeze(0)
        # PyTorch's convention is that True entries are blocked.
        causal_mask = torch.triu(
            torch.ones(
                sequence_length,
                sequence_length,
                dtype=torch.bool,
                device=token_ids.device,
            ),
            diagonal=1,
        )
        hidden = self.decoder(hidden, mask=causal_mask)
        hidden = self.final_norm(hidden)
        return self.lm_head(hidden)


# Print the requested 8-by-8 mask and verify the [batch, length, vocab] logits shape.
mask_8 = torch.triu(torch.ones(8, 8, dtype=torch.bool), diagonal=1)
print(mask_8.int())
lm_model = TinyCharLM(
    vocab_size=len(itos), d_model=32, nhead=4, num_layers=2, max_len=SEQ_LEN
)
lm_model.eval()
with torch.no_grad():
    example_logits = lm_model(x_lm[:2])
print("logits shape:", tuple(example_logits.shape))


tensor([[0, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 1, 1, 1, 1],
        [0, 0, 0, 0, 0, 1, 1, 1],
        [0, 0, 0, 0, 0, 0, 1, 1],
        [0, 0, 0, 0, 0, 0, 0, 1],
        [0, 0, 0, 0, 0, 0, 0, 0]], dtype=torch.int32)
logits shape: (2, 64, 58)


<div dir="rtl">

### توضیح بخش ۲ـج

- `True` در ماسک یعنی چه؟ یعنی آن خانه از attention مسدود است؛ توکن در آن موقعیت اجازه ندارد به توکن‌های آینده نگاه کند. قطر اصلی و خانه‌های پایین‌تر مجاز می‌مانند.
- Encoder را انتخاب کردید یا Decoder؟ اینجا یک پیاده‌سازی جمع‌وجورِ decoder-style انتخاب شده است: همان لایهٔ Transformer با causal mask روی یک جریان توکن اجرا می‌شود، ولی cross-attention به حافظهٔ encoder ندارد. این دقیقاً مناسب پیش‌بینی توکن بعدی است.
- Decoder نسبت به Encoder چه زیرماژول اضافه‌ای دارد؟ در معماری کامل TransformerDecoder، لایهٔ masked self-attention دارد و در حالت encoder-decoder یک cross-attention نیز به حافظهٔ encoder اضافه می‌کند. این نسخهٔ تک‌جریان فقط از causal self-attention استفاده می‌کند و حافظهٔ بیرونی ندارد.

</div>


<div dir="rtl">

## بخش ۲ـ د. آموزش کوتاه و تولید متن

باید انجام دهید:
- چند ده step با CrossEntropyLoss
- تابع تولید: پیشوند → توکن بعدی → اضافه کردن → تکرار
- شروع از `ROMEO:` یا `First Citizen:`
- حدود ۸۰ تا ۱۲۰ نویسه چاپ کنید

</div>


In [14]:
# Optimize the same causal model for a few dozen next-character steps.
lm_model.to(DEVICE)
lm_optimizer = torch.optim.AdamW(lm_model.parameters(), lr=3e-3)
lm_criterion = nn.CrossEntropyLoss()
lm_steps = 50
lm_losses = []

for step in range(lm_steps):
    batch_indices = torch.randint(0, x_lm.shape[0], (32,))
    lm_optimizer.zero_grad(set_to_none=True)
    logits = lm_model(x_lm[batch_indices].to(DEVICE))
    targets = y_lm[batch_indices].to(DEVICE)
    loss = lm_criterion(logits.reshape(-1, len(itos)), targets.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(lm_model.parameters(), max_norm=1.0)
    lm_optimizer.step()
    lm_losses.append(float(loss.item()))
    if (step + 1) % 10 == 0:
        print(f"step {step + 1:02d}/{lm_steps} | loss={lm_losses[-1]:.4f}")

print("final LM loss:", round(lm_losses[-1], 4))


step 10/50 | loss=3.4353


step 20/50 | loss=3.1580


step 30/50 | loss=3.0275


step 40/50 | loss=2.8518


step 50/50 | loss=2.7752
final LM loss: 2.7752


In [15]:
def generate(model, prefix: str, max_new_tokens: int = 120) -> str:
    """Greedily append one character at a time using only the visible prefix."""
    model.eval()
    fallback_id = next(iter(stoi.values()))
    token_ids = [stoi.get(character, fallback_id) for character in prefix]
    if not token_ids:
        token_ids = [fallback_id]
    context = torch.tensor([token_ids[-model.max_len :]], dtype=torch.long)
    generated = prefix

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(context.to(DEVICE))
            next_id = int(logits[0, -1].argmax().item())
            generated += itos[next_id]
            context = torch.cat(
                [context, torch.tensor([[next_id]], dtype=torch.long)], dim=1
            )[:, -model.max_len :]
    return generated


generated_text = generate(lm_model, "ROMEO:", max_new_tokens=100)
print(generated_text)


ROMEO:
S the the t t athe the t the t t at se are the t t the the t t t t t t t t t t t t t t t t t t t t 


<div dir="rtl">

### توضیح بخش ۲ـ د

- loss چقدر کم شد؟ در این اجرا از حدود ۳٫۴ در گام‌های اولیه به ۲٫۷۸ در گام ۵۰ رسید؛ یعنی کاهش محسوس، اما نه کمینه‌شدن loss. این مقدار به seed، batch و تعداد گام وابسته است و مقدار چاپ‌شده در خروجی سلول آموزش ملاک همین اجراست.
- متن تولیدی چه الگویی را تا حدی یاد گرفت و چه چیزی را بلد نیست؟ از پیشوند `ROMEO:` شروع می‌کند و احتمالاً قالب نام‌گذاری، حروف، فاصله و newlineهای متن Shakespeare را تقلید می‌کند. اما چون داده و آموزش کوچک است، معنا، پیوستگی بلندمدت، نام‌ها و اتفاق داستان را به‌خوبی نمی‌داند.

</div>


<div dir="rtl">

## بخش ۲ـ ه. یک آزمایش کوچک و جمع‌بندی

فقط **یکی** از سه آزمایش PDF را انجام دهید. بعد به دو سؤال آخر جواب دهید.

</div>


In [16]:
# Chosen experiment: keep the same causal architecture but remove positional
# information for a short run.  Zeroing this table makes the ablation explicit.
def run_position_ablation(use_positions, steps=30):
    torch.manual_seed(1234)
    experiment_model = TinyCharLM(
        vocab_size=len(itos), d_model=32, nhead=4, num_layers=2, max_len=SEQ_LEN
    )
    if not use_positions:
        with torch.no_grad():
            experiment_model.position_embedding.weight.zero_()
    experiment_optimizer = torch.optim.AdamW(experiment_model.parameters(), lr=3e-3)
    experiment_criterion = nn.CrossEntropyLoss()
    batch_generator = torch.Generator().manual_seed(5678)
    history = []
    for _ in range(steps):
        batch_indices = torch.randint(
            0, x_lm.shape[0], (16,), generator=batch_generator
        )
        experiment_optimizer.zero_grad(set_to_none=True)
        logits = experiment_model(x_lm[batch_indices])
        loss = experiment_criterion(
            logits.reshape(-1, len(itos)), y_lm[batch_indices].reshape(-1)
        )
        loss.backward()
        experiment_optimizer.step()
        history.append(float(loss.item()))
    return history


with_position_loss = run_position_ablation(True)
without_position_loss = run_position_ablation(False)
print(
    "with positional embedding    | final loss:",
    round(with_position_loss[-1], 4),
)
print(
    "without positional embedding | final loss:",
    round(without_position_loss[-1], 4),
)


with positional embedding    | final loss: 3.0673
without positional embedding | final loss: 2.83


<div dir="rtl">

### توضیح بخش ۲ـ ه

آزمایش انتخابی: حذف embedding موقعیت. در کد، جدول موقعیتِ مدل آزمایشی صفر می‌شود و دو مدل با داده و گام‌های یکسان کوتاه آموزش می‌بینند. در این اجرا loss نهاییِ مدلِ دارای موقعیت ۳٫۰۶۷۳ و مدلِ بدون موقعیت ۲٫۸۳۰۰ شد؛ یعنی در ۳۰ گام کوتاه، حذف موقعیت کمی بهتر به نظر رسید. این نتیجهٔ قطعی نیست: مدل بدون موقعیت هنوز می‌تواند از محتوای توکن‌ها و الگوهای محلی استفاده کند، اما سیگنال ترتیب و وابستگی‌های دور را ندارد. تعداد گام کم، batch متفاوت و نویز آموزش می‌تواند این اختلاف را جابه‌جا کند.

**سؤال‌ها**

1. پروژهٔ ۱ در حال طبقه‌بندی یک سند کامل است و از هر توکن می‌تواند به کل زمینهٔ موجود دسترسی داشته باشد؛ causal mask اطلاعات معتبر را پنهان می‌کند. در پروژهٔ ۲، خروجی هر موقعیت باید فقط از پیشوند و گذشته ساخته شود. اگر آموزش اجازهٔ دیدن آینده بدهد، توکن بعدی از پاسخ لو می‌رود و تولید autoregressive با آموزش ناسازگار می‌شود.
2. در pretraining خودنظارتی، مدل از متن خام برای یادگیری توزیع توکن بعدی (یا هدف مشابه) استفاده می‌کند و برچسب انسانی لازم ندارد؛ این دانش عمومی زبان سپس می‌تواند به کارهای دیگر منتقل شود. در fine-tune روی موضوع خبر، همان مدل با نمونه‌های برچسب‌دار و تابع هزینهٔ طبقه‌بندی روی دو کلاس تنظیم می‌شود تا تصمیم مخصوص همان وظیفه را بگیرد. در بخش ۱ِ این notebook، embedding مستقیماً و از صفر برای طبقه‌بندی آموزش دیده می‌شود، نه به‌صورت pretraining.

</div>


<div dir="rtl">

# چک‌لیست قبل از ارسال

- [x] پروژهٔ ۱ روی حداکثر ۲۵۰+۱۰۰ سند از دو کلاس خواسته‌شده اجرا شده است.
- [x] BoW، میانگین embedding و Encoder گزارش شده‌اند.
- [x] دو سند اشتباه BoW و یک جفت جملهٔ دلخواه آمده است.
- [x] چند جفت skip-gram ساخته شده است.
- [x] پروژهٔ ۲ روی برش کوچک Tiny Shakespeare است.
- [x] BERT و GPT-2 فقط برای مشاهدهٔ دو خط استفاده شده‌اند.
- [x] یک مثال ورودی/هدف شیفت‌شده چاپ شده است.
- [x] ماسک علّی چاپ شده و یک متن کوتاه تولید شده است.
- [x] همهٔ سلول‌های «توضیح این بخش» پر شده‌اند.

</div>
